In [ ]:
# Day 5: Similarity Visualization & Scoring

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# -----------------------------
# Load datasets (absolute paths)
# -----------------------------
employees = pd.read_csv(r"C:/Users/KUMAR/Desktop/staffing_copilot/data/processed/employees.csv")
projects  = pd.read_csv(r"C:/Users/KUMAR/Desktop/staffing_copilot/data/processed/projects.csv")
kaggle_employees = pd.read_csv(r"C:/Users/KUMAR/Desktop/staffing_copilot/data/raw/Employee.csv")

merged = employees.merge(
    kaggle_employees[['JobRole','Department','Salary','Attrition']],
    left_on='role',
    right_on='JobRole',
    how='left'
)

# -----------------------------
# Prepare text fields
# -----------------------------
merged['Department'] = merged['Department'].fillna("Unknown")
merged['skills'] = merged['skills'].fillna("[]")

merged['profile_text'] = (
    merged['role'].astype(str) + " | " +
    merged['skills'].astype(str) + " | " +
    merged['Department'].astype(str)
)

projects['requirement_text'] = (
    projects['required_roles'].astype(str) + " | " +
    projects['required_skills'].astype(str)
)

# -----------------------------
# Generate embeddings
# -----------------------------
model = SentenceTransformer('all-MiniLM-L6-v2')
employee_embeddings = model.encode(merged['profile_text'].tolist())
project_embeddings = model.encode(projects['requirement_text'].tolist())

similarity_matrix = cosine_similarity(project_embeddings, employee_embeddings)

# -----------------------------
# Heatmap visualization
# -----------------------------
plt.figure(figsize=(12,8))
sns.heatmap(similarity_matrix, cmap="viridis")
plt.title("Project vs Employee Similarity Heatmap")
plt.xlabel("Employees")
plt.ylabel("Projects")
plt.show()

# -----------------------------
# Weighted scoring logic
# -----------------------------
# Normalize salary (lower salary = better score)
merged['salary_score'] = 1 - (merged['Salary'] / merged['Salary'].max())

# Attrition: employees with 'Yes' attrition risk get lower score
merged['attrition_score'] = merged['Attrition'].apply(lambda x: 0.2 if x=="Yes" else 1.0)

# Final weighted score = similarity * salary_score * attrition_score
def top_matches_with_scoring(project_idx, top_n=5):
    scores = similarity_matrix[project_idx]
    weighted_scores = scores * merged['salary_score'] * merged['attrition_score']
    top_indices = weighted_scores.argsort()[-top_n:][::-1]
    print(f"\nTop {top_n} matches for Project {projects.iloc[project_idx]['project_name']}:")
    for i in top_indices:
        print(f"Employee {merged.iloc[i]['name']} ({merged.iloc[i]['role']}) "
              f"- Similarity: {scores[i]:.3f}, Weighted: {weighted_scores[i]:.3f}")

# Example: show matches for first 3 projects
for p in range(3):
    top_matches_with_scoring(p)
